# Lennard-Jones MD

This notebook compares NVE energy drift with Langevin NVT temperature control in Lennard-Jones reduced units.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.core import Cell
from mlx_atomistic.diagnostics import summarize_md_result
from mlx_atomistic.md import LangevinThermostat, LennardJonesPotential, SimulationConfig, simulate_nve, simulate_nvt
from mlx_atomistic.neighbors import NeighborListManager
from mlx_atomistic.runtime import get_runtime_info

get_runtime_info()


In [ ]:
positions = np.array([
    [1.0, 1.0, 1.0],
    [2.3, 1.0, 1.0],
    [1.0, 2.3, 1.0],
    [2.3, 2.3, 1.0],
], dtype=np.float32)

velocities = np.array([
    [0.02, 0.01, 0.0],
    [-0.02, 0.01, 0.0],
    [0.02, -0.01, 0.0],
    [-0.02, -0.01, 0.0],
], dtype=np.float32)

cell = Cell.cubic(6.0)
potential = LennardJonesPotential(cutoff=2.5)
config = SimulationConfig(dt=0.002, steps=250, sample_interval=10)


In [ ]:
nve = simulate_nve(
    positions,
    velocities,
    cell=cell,
    force_terms=potential,
    neighbor_manager=NeighborListManager(cell, cutoff=2.5),
    config=config,
)

nvt = simulate_nvt(
    positions,
    velocities,
    cell=cell,
    force_terms=potential,
    neighbor_manager=NeighborListManager(cell, cutoff=2.5),
    config=config,
    thermostat=LangevinThermostat(temperature=1.0, friction=1.0, seed=7),
)

summarize_md_result(nve), summarize_md_result(nvt)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.array(nve.energy_drift), label="NVE ΔE")
plt.plot(np.array(nvt.total_energy) - np.array(nvt.total_energy)[0], label="NVT ΔE")
plt.xlabel("step")
plt.ylabel("energy / ε")
plt.legend()
plt.tight_layout()


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.array(nve.temperature), label="NVE T")
plt.plot(np.array(nvt.temperature), label="NVT T")
plt.axhline(nvt.target_temperature, color="black", linestyle="--", linewidth=1, label="target T")
plt.xlabel("step")
plt.ylabel("temperature / (ε/k_B)")
plt.legend()
plt.tight_layout()


In [ ]:
plt.figure(figsize=(7, 3))
plt.step(np.arange(len(np.array(nve.rebuild_count))), np.array(nve.rebuild_count), where="post", label="NVE rebuilds")
plt.step(np.arange(len(np.array(nvt.rebuild_count))), np.array(nvt.rebuild_count), where="post", label="NVT rebuilds")
plt.xlabel("step")
plt.ylabel("neighbor-list rebuild count")
plt.legend()
plt.tight_layout()
